# Porcentaje de condenas por violencia de género

## Visualizaciones objetivo

A fecha de análisis, este dataset se emplea en visualizaciones "paralelas" a las del dataset de [órdenes de protección](./n4-ordenes_proteccion.ipynb).

- Para DxVxM (España)

    - [dxvxm-espanya-v_2_4-esp](https://app.flourish.studio/visualisation/1252526/edit): grid (1 chart por CCAA + media España) de bar charts con el % de condenas (Y) por año (X)

- Para DxVxD (Cataluña)

    - [dxvxm-catalunya-v_2_4-cat](https://app.flourish.studio/visualisation/1153112/edit): grouped bar chart de la media de Cat vs Es del % de condenas (Y) por año (X)

- Para Lleida:

    - [dxvxm-lleida-v_3_3-esp](https://app.flourish.studio/visualisation/19011779/edit) ([-cat](https://app.flourish.studio/visualisation/18990183/edit)): 3 bar charts (provincia de Lleida, Cat, Es) con el % de condenas (Y) por año (X)
    
    - [dxvxm-lleida-v_3_4-esp](https://app.flourish.studio/visualisation/19012014/edit) ([-cat](https://app.flourish.studio/visualisation/18990189/edit)): grid similar al dxvxm-lleida-v_3_3-esp pero con la provincia vs sus partidos judiciales

        - Hay una alta variabilidad en algunos partidos debido a que hay pocos casos. Se podría reemplazar por un solo bar chart que, para cada partido judicial, saque la media entre todos los años, o se pueden usar periodos de por ejemplo 5 años en vez de ir año en año.

        - También se podría enfocar de forma relativa a Lleida, ya que la evolución de la provincia ya se ve en el 3_3. Esto sería que cada año la provincia fuera el 100% y para cada partido se viera la diferencia, ya sea multiplicada o sumada (por ejemplo TREMP en 2023 +12% que la provincia).

In [ ]:
import pandas as pd

from pathlib import Path

PATH_DATA = Path("../data")
PATH_INPUT_MANUAL = PATH_DATA / "cgpj_manual"

## Descarga manual

Indicaciones:

1. Acceder a https://www.poderjudicial.es/cgpj/es/Temas/Estadistica-Judicial/Estadistica-por-temas/Datos-penales--civiles-y-laborales/Violencia-domestica-y-Violencia-de-genero/Datos-sobre-Violencia-sobre-la-mujer-en-la-estadistica-del-CGPJ/

2. pulsar "Datos". Seleccionar el año YYYY que nos interesa y periodo "Anual". Pulsar "Buscar".

3. Hacer las siguientes descargas y búsquedas en los archivos (se recomienda buscar con "CTRL F"), rellenando la plantilla correspondiente en cada paso:

    - Para actualizar cualquiera de los proyectos (España, Cataluña o Lleida):

        - Descargar el PDF "Violencia sobre la mujer - Anual YYYY".
        
            - Buscar "Total Sentencias dictadas en el ámbito de la Violencia de Género". Debe haber una mini tabla con los datos para ese año. Nos interesa el valor de la columnas "%" para el órgano "JVM".

                - Alerta: en 2022 y 2023, el nombre era parecido, pero con una coma en medio. Eso puede dificultar la búsqueda. Igualmente, se puede acceder a la tabla desde el índice.

        - Descargar el Excel "Comunidades - Anual YYYY".
        
            - Para cada CCAA (o solo para Cataluña, según el proyecto), ir a su página en el Excel y buscar "Porcentaje Sentencias Condenatorias". Hay varias opciones, pero nos interesa la primera (la de más arriba). Hay que coger el porcentaje en la columna del año que nos interesa (no la del año anterior).

                - Las instrucciones originales son similares, pero dan a intuir que había alguna macro que hacía este trabajo.

            - Pendiente: ver si cada año el formato es el mismo. Si lo es, el Excel se puede usar como entrada sin buscar los datos ni ponerlos en una plantilla


    - Solamente para el de Lleida:

        - Descargar los Excels "Violencia sobre la Mujer Juzgados por Provincias - Anual YYYY" y "Violencia sobre la Mujer Juzgados por Partido Judicial - Anual YYYY".

            - Ir a la página "% de Condenas" y coger los valores de la columna "% condenas entre los  enjuiciados" que nos interesen.

                - Para Lleida, hay que coger:

                    - En el de provincias, "Lleida"

                    - En el de partidos judiciales, "TREMP", "BALAGUER", "CERVERA", "LLEIDA", "LA SEU D'URGELL", "VIELHA E MIJARAN" i "SOLSONA"
                    

### ¿Cada año los PDFs y Excels tienen el mismo formato?

Primero es interesante ver si el nombre y el enlace se mantienen.

Documento 1: Violencia sobre la mujer - Anual (PDF)

| Año | Nombre | Enlace | ¿Tabla contiene texto exacto? |
|---|---|---|---|
| 2025 | Violencia sobre la mujer - Anual 2025 | https://www.poderjudicial.es/stfls/ESTADISTICA/DOCUMENTOSCGPJ/Violencia%20sobre%20la%20Mujer%20-%20Informe%20anual%202025.pdf | sí |
| 2024 | Violencia sobre la mujer - Anual 2024 | https://www.poderjudicial.es/stfls/ESTADISTICA/DOCUMENTOSCGPJ/Violencia%20sobre%20la%20Mujer%20-%20Informe%20anual%202024.docx | sí |
| 2023 | Violencia sobre la mujer - Año 2023 | https://www.poderjudicial.es/stfls/ESTADISTICA/DOCUMENTOSCGPJ/Violencia%20sobre%20la%20mujer%20-%20A%C3%B1o%202023.pdf | no |
| 2022 | Violencia sobre la mujer - Anual 2022 | https://www.poderjudicial.es/stfls/ESTADISTICA/DOCUMENTOSCGPJ/Violencia%20sobre%20la%20mujer%20-%20A%C3%B1o%202022.pdf | no |
| 2021 | Violencia sobre la mujer - Anual 2021 | https://www.poderjudicial.es/stfls/ESTADISTICA/DOCUMENTOSCGPJ/Violencia%20sobre%20la%20mujer%20%20Informe%20Anual%20de%202021.pdf | sí |

Como se puede observar, el nombre cambia el 2023 y el enlace no está estandarizado (no se puede hacer ni una sola pareja). Además, ni tan solo el formato se mantiene, ya que en 2024 es un .docx.

El valor objetivo sí que está mejor estandarizado, aunque a veces se cuela una coma. Se puede plantear la extracción de alguna forma automática.

Documento 2: Comunidades - Anual

| Año | Nombre | Enlace |
|---|---|---|
| 2025 |  Comunidades - Anual 2025 | https://www.poderjudicial.es/stfls/ESTADISTICA/DOCUMENTOSCGPJ/Comunidades%20-%20Anual%202025.xlsx |
| 2024 | Comunidades - Anual 2024 | https://www.poderjudicial.es/stfls/ESTADISTICA/DOCUMENTOSCGPJ/Comunidades.%20Anual%202024.xlsx |
| 2023 | Comunidades - Año 2023 | https://www.poderjudicial.es/stfls/ESTADISTICA/DOCUMENTOSCGPJ/Comunidades%20-%20A%C3%B1o%202023.xlsx |
| 2022 |  Comunidades - Año 2022 | https://www.poderjudicial.es/stfls/ESTADISTICA/DOCUMENTOSCGPJ/Comunidades%20-%20%20A%C3%B1o%202022.xlsx |
| 2021 | Comunidades - Año 2021 | https://www.poderjudicial.es/stfls/ESTADISTICA/DOCUMENTOSCGPJ/Comunidades%20-%20A%C3%B1o%202021.xlsx |

El enlace estaba más estandarizado antes que ahora.

La estructura de los Excels sí que es la misma año tras año.

Con los Excels por provincia y partido judicial sucede algo similar.

Es conveniente que no se "hardcodee" ninguna posición de ninguna celda o, si en algún momento cambia la estructura en partidos judiciales, se podría romper el esquema.